In [3]:
import pandas as pd

# 1. Load your raw WFP file (skip row 1 metadata tags)
df = pd.read_csv("../data/raw/wfp_food_prices_nga.csv", skiprows=[1], low_memory=False)

# 2. Standardize dates to YYYY-MM-01 and filter to Jan 2020 - Dec 2025
df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()
df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-01")].copy()

# 3. Convert 0.5 KG prices to 1 KG
df["price_per_kg"] = df.apply(
    lambda r: float(r["price"]) * 2.0 if str(r["unit"]).strip().upper() == "0.5 KG" else float(r["price"]),
    axis=1,
)

# 4. Compute monthly average price per kg for Tomatoes and Onions
tomatoes = (
    df[df["commodity"] == "Tomatoes"]
    .groupby("date")["price_per_kg"]
    .mean()
    .rename("tomato_price_ngn")
)

onions = (
    df[df["commodity"] == "Onions"]
    .groupby("date")["price_per_kg"]
    .mean()
    .rename("onion_price_ngn")
)

# 5. Reindex to all 72 months (Jan 2020 - Dec 2025) and fill the 8 missing months
full_months = pd.date_range("2020-01-01", "2025-12-01", freq="MS")
food_clean = pd.DataFrame(index=full_months).join([tomatoes, onions])
food_clean.index.name = "date"
food_clean = food_clean.interpolate(method="linear").ffill().bfill().round(2).reset_index()

# 6. Save your cleaned food prices into data/processed/
food_clean.to_csv("../data/processed/clean_food_prices.csv", index=False)

print("Done! Saved data/processed/clean_food_prices.csv")
print("Total rows:", len(food_clean))
print(food_clean.head(5))

Done! Saved data/processed/clean_food_prices.csv
Total rows: 72
        date  tomato_price_ngn  onion_price_ngn
0 2020-01-01            142.47           216.22
1 2020-02-01            113.68           205.68
2 2020-03-01            126.68           161.25
3 2020-04-01            135.20           170.20
4 2020-05-01            131.09           195.66
